# Skin Cancer Detection — GPU Ensemble Training (Google Colab T4)

This notebook clones your GitHub repository (`debayudh482/skin-cancer-detection`) and trains the **3-Model Deep Ensemble (`EfficientNet-B0 + DenseNet121 + MobileNetV2`) + 4x TTA + Stage-1 Binary Triage** on a free Google Colab T4 GPU in ~5–8 minutes.

- **Runtime:** Go to `Runtime -> Change runtime type -> T4 GPU` before running.
- **Dataset:** Upload `Skin_ISIC.zip` (or load **HAM10000 / ISIC 2018** via Kaggle API for **85%+ multi-class accuracy**).

In [ ]:
!nvidia-smi
!git clone https://github.com/debayudh482/skin-cancer-detection.git
%cd skin-cancer-detection
!pip install -q -r requirements.txt

### Step 1: Load Dataset
Mount Google Drive (where `Skin_ISIC.zip` is uploaded) OR download HAM10000 from Kaggle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update the path below if your zip is in a subfolder on Drive
!mkdir -p data
!unzip -q "/content/drive/MyDrive/Skin_ISIC.zip" -d data/
!python -m src.prepare_data

### Step 2: Train 3 Diverse Backbones on GPU (~1–2 min each on T4)

In [ ]:
!python -m src.train --model efficientnet_b0 --unfreeze all --stage1-epochs 4 --stage2-epochs 18 --skip-eval
!python -m src.train --model densenet121 --unfreeze all --stage1-epochs 4 --stage2-epochs 18 --skip-eval
!python -m src.train --model mobilenet_v2 --unfreeze all --stage1-epochs 4 --stage2-epochs 18 --skip-eval

### Step 3: Train Stage-1 Binary Triage + Evaluate 3-Model Ensemble with 4x TTA

In [ ]:
!python -m src.train_binary --model efficientnet_b0 --stage1-epochs 4 --stage2-epochs 12
!python -m src.ensemble --checkpoints checkpoints/efficientnet_b0_best.pth checkpoints/densenet121_best.pth checkpoints/mobilenet_v2_best.pth

### Step 4: Inspect Generated Plots

In [ ]:
from IPython.display import Image, display
for img in ['outputs/binary_triage_evaluation.png', 'outputs/confusion_matrix_val.png', 'outputs/roc_curves_val.png', 'outputs/gradcam_samples.png']:
    display(Image(filename=img))